In [6]:
import scipy.io
import netCDF4 as nc
import numpy as np
import h5py
import os
from pathlib import Path

# ── Configure your paths here ──────────────────────────────────────────────────
DATA_DIR   = Path("/home/jovyan/Society_of_Bouy_Cowboys/Data/Ship_Data")
OUTPUT_DIR = Path("/home/jovyan/Society_of_Bouy_Cowboys/Data/Ship_Data")
# ──────────────────────────────────────────────────────────────────────────────

VALID_DTYPES = {'S1','i1','u1','i2','u2','i4','u4','i8','u8','f4','f8'}

DTYPE_FALLBACK = {
    'float128': 'f8', 'float96': 'f8', 'float64': 'f8', 'float32': 'f4',
    'int64': 'i8',   'int32': 'i4',   'int16': 'i2',   'int8': 'i1',
    'uint64': 'u8',  'uint32': 'u4',  'uint16': 'u2',  'uint8': 'u1',
    'bool': 'u1',
}

def is_hdf5(filepath):
    with open(filepath, 'rb') as f:
        return f.read(4) == b'\x89HDF'

def sanitize_name(name):
    return str(name).replace('-', '_').replace(' ', '_').replace('.', '_').strip('_')

def resolve_dtype(data):
    """Map numpy dtype to a netCDF4-compatible dtype string."""
    dtype_str = data.dtype.str.lstrip('|<>=')  # e.g. 'f8', 'i4'
    if dtype_str in VALID_DTYPES:
        return dtype_str
    name = data.dtype.name  # e.g. 'float64'
    if name in DTYPE_FALLBACK:
        return DTYPE_FALLBACK[name]
    return None  # unsupported

def write_array(ds, var_name, data):
    """Write a single numpy array to the NetCDF dataset."""
    var_name = sanitize_name(var_name)
    data = np.array(data)

    # Flatten object arrays of numbers
    if data.dtype.kind == 'O':
        try:
            data = np.array(data.tolist(), dtype=float)
        except (ValueError, TypeError):
            print(f"  Skipping '{var_name}': object array could not be cast to float")
            return

    dtype_str = resolve_dtype(data)
    if dtype_str is None:
        print(f"  Skipping '{var_name}': unsupported dtype '{data.dtype}'")
        return

    # Squeeze out size-1 wrapper dimensions (common in scipy.io output)
    data = np.squeeze(data)
    if data.ndim == 0:
        data = data.reshape(1)

    # Create dimensions
    dims = []
    for i, size in enumerate(data.shape):
        dim_name = f"{var_name}_dim{i}"
        if dim_name not in ds.dimensions:
            ds.createDimension(dim_name, size)
        dims.append(dim_name)

    var = ds.createVariable(var_name, dtype_str, tuple(dims), zlib=True)
    var[:] = data
    print(f"  ✓ '{var_name}' | shape: {data.shape} | dtype: {dtype_str}")

def unpack_and_write(ds, key, value):
    """Handle plain arrays, structured arrays, and nested structs."""
    key = sanitize_name(key)

    # Structured array (MATLAB struct/table) — unpack each field
    if value.dtype.names:
        print(f"  Unpacking struct '{key}' with fields: {value.dtype.names}")
        for field in value.dtype.names:
            field_data = value[field]
            # Squeeze the outer wrapper scipy adds
            if field_data.ndim > 1:
                field_data = np.squeeze(field_data)
            # Recurse if nested struct
            if field_data.dtype.names:
                unpack_and_write(ds, f"{key}_{field}", field_data)
            else:
                write_array(ds, f"{key}_{field}", field_data)
    else:
        write_array(ds, key, value)

def convert(mat_path: Path, nc_path: Path):
    mat_path = Path(mat_path)
    nc_path  = Path(nc_path)

    if not mat_path.exists():
        raise FileNotFoundError(f"File not found: {mat_path}")

    print(f"\nInput:  {mat_path}")
    print(f"Output: {nc_path}\n")

    ds = nc.Dataset(nc_path, 'w', format='NETCDF4')
    ds.source = mat_path.name

    if is_hdf5(mat_path):
        print("Detected: MATLAB v7.3 (HDF5)\n")
        with h5py.File(mat_path, 'r') as f:
            for key in f.keys():
                if key.startswith('#'):
                    continue
                try:
                    write_array(ds, key, np.array(f[key]))
                except Exception as e:
                    print(f"  Skipping '{key}': {e}")
    else:
        print("Detected: Legacy MATLAB (<= v7.2)\n")
        mat = scipy.io.loadmat(mat_path, squeeze_me=True, struct_as_record=True)
        for key, value in mat.items():
            if key.startswith('_'):
                continue
            if isinstance(value, np.ndarray):
                unpack_and_write(ds, key, value)
            else:
                print(f"  Skipping '{key}': type '{type(value).__name__}' not supported")

    ds.close()
    print(f"\n✓ Saved: {nc_path}")


# ── Convert all .mat files in DATA_DIR ─────────────────────────────────────────
mat_files = list(DATA_DIR.glob("*.mat"))

if not mat_files:
    print(f"No .mat files found in {DATA_DIR}")
else:
    for mat_file in mat_files:
        out_file = OUTPUT_DIR / mat_file.with_suffix('.nc').name
        convert(mat_file, out_file)


Input:  /home/jovyan/Society_of_Bouy_Cowboys/Data/Ship_Data/Latest_Underway_Data_10min_avg.mat
Output: /home/jovyan/Society_of_Bouy_Cowboys/Data/Ship_Data/Latest_Underway_Data_10min_avg.nc

Detected: Legacy MATLAB (<= v7.2)

  Unpacking struct 'S' with fields: ('time', 'dnum', 'met_ptu307', 'met_met4a_fwdmast', 'wind_mast_port_true', 'wind_mast_stbd_true', 'wind_gill_fwdmast_true', 'rad_psp_pir', 'tsg_sbe45_fwd', 'tsg_sbe45_fwd_2', 'ins_seapath_position_GPRMC', 'ins_seapath_position_PSXN23', 'thermo_pyrometer_ct15', 'best_wind')
  ✓ 'S_time' | shape: (2881,) | dtype: f8
  ✓ 'S_dnum' | shape: (2881,) | dtype: f8
  Skipping 'S_met_ptu307': object array could not be cast to float
  Skipping 'S_met_met4a_fwdmast': object array could not be cast to float
  Skipping 'S_wind_mast_port_true': object array could not be cast to float
  Skipping 'S_wind_mast_stbd_true': object array could not be cast to float
  Skipping 'S_wind_gill_fwdmast_true': object array could not be cast to float
  Skippi

In [10]:
import xarray as xr
ship_df = xr.open_dataset("/home/jovyan/Society_of_Bouy_Cowboys/Data/Ship_Data/ShipData_2019.nc")
ship_df

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*